# Phase 2 – MLflow Model Lifecycle Management

**With SMOTE for class imbalance handling.**




## 1. Imports & Dependencies

In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score
)
from imblearn.over_sampling import SMOTE

import statsmodels.api as sm

import mlflow
import mlflow.sklearn
import mlflow.statsmodels
from mlflow.models.signature import infer_signature

print('All imports OK')
print('MLflow version:', mlflow.__version__)

All imports OK
MLflow version: 3.11.1


## 2. MLflow Setup

In [4]:
TRACKING_URI = 'sqlite:///mlflow.db'
EXPERIMENT_NAME = 'Phishing_Detection_Phase2_SMOTE'

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print('Tracking URI :', mlflow.get_tracking_uri())
print('Experiment   :', EXPERIMENT_NAME)

2026/04/19 07:34:10 INFO mlflow.tracking.fluent: Experiment with name 'Phishing_Detection_Phase2_SMOTE' does not exist. Creating a new experiment.


Tracking URI : sqlite:///mlflow.db
Experiment   : Phishing_Detection_Phase2_SMOTE


## 3. Data Preparation + SMOTE

In [5]:
DATASET_VERSION = 'v1'

df = pd.read_csv('phishing_dataset.csv')
print('Dataset shape:', df.shape)
print('\n=== Label Distribution (BEFORE SMOTE) ===')
print(df['label'].value_counts())
print('\nClass ratio:')
print(df['label'].value_counts(normalize=True))

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')


print('\n=== Applying SMOTE to Training Set ===')
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f'Train shape AFTER SMOTE: {X_train_smote.shape}')
print('\n=== Label Distribution (AFTER SMOTE) ===')
print(pd.Series(y_train_smote).value_counts())
print('\nClass ratio:')
print(pd.Series(y_train_smote).value_counts(normalize=True))
print(f'\nTest set (unchanged): {X_test.shape}')

Dataset shape: (524846, 9)

=== Label Distribution (BEFORE SMOTE) ===
label
0    517897
1      6949
Name: count, dtype: int64

Class ratio:
label
0    0.98676
1    0.01324
Name: proportion, dtype: float64

Train: (419876, 8), Test: (104970, 8)

=== Applying SMOTE to Training Set ===
Train shape AFTER SMOTE: (828634, 8)

=== Label Distribution (AFTER SMOTE) ===
label
0    414317
1    414317
Name: count, dtype: int64

Class ratio:
label
0    0.5
1    0.5
Name: proportion, dtype: float64

Test set (unchanged): (104970, 8)


## 4. Helper – Plot & Save Confusion Matrix

In [6]:
def plot_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Phishing'],
                yticklabels=['Legit', 'Phishing'])
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    return filename

## 5. Run 1 – Logistic Regression (with SMOTE)

In [7]:
with mlflow.start_run(run_name='LogisticRegression_SMOTE') as run_lr:
    # Train on SMOTE-balanced data
    C_val = 1.0
    max_iter_val = 1000
    solver_val = 'lbfgs'

    clf = LogisticRegression(C=C_val, max_iter=max_iter_val, solver=solver_val)
    clf.fit(X_train_smote, y_train_smote)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    # Metrics on TEST set
    acc   = accuracy_score(y_test, y_pred)
    f1    = f1_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, zero_division=0)
    rec   = recall_score(y_test, y_pred, zero_division=0)
    auc   = roc_auc_score(y_test, y_proba)

    mlflow.log_param('model_type',     'Logistic Regression')
    mlflow.log_param('C',              C_val)
    mlflow.log_param('max_iter',       max_iter_val)
    mlflow.log_param('solver',         solver_val)
    mlflow.log_param('balancing',      'SMOTE')
    mlflow.log_param('train_size_before', X_train.shape[0])
    mlflow.log_param('train_size_after', X_train_smote.shape[0])

    mlflow.log_metric('accuracy',  acc)
    mlflow.log_metric('f1_score',  f1)
    mlflow.log_metric('precision', prec)
    mlflow.log_metric('recall',    rec)
    mlflow.log_metric('roc_auc',   auc)

    mlflow.set_tag('Algorithm',       'Logistic Regression')
    mlflow.set_tag('Dataset_Version', DATASET_VERSION)
    mlflow.set_tag('Balancing',       'SMOTE')

    cm_path = plot_confusion_matrix(y_test, y_pred,
                                    'Confusion Matrix – Logistic Regression (SMOTE)',
                                    'lr_cm_smote.png')
    mlflow.log_artifact(cm_path)

    sig = infer_signature(X_train_smote, clf.predict(X_train_smote))
    mlflow.sklearn.log_model(clf, 'logistic_regression_model', signature=sig)

    print(f'LogReg (SMOTE)  |  acc={acc:.4f}  f1={f1:.4f}  recall={rec:.4f}  auc={auc:.4f}')

2026/04/19 07:34:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 07:34:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LogReg (SMOTE)  |  acc=0.5010  f1=0.0404  recall=0.7942  auc=0.6857


### Confusion Matrix – Logistic Regression (SMOTE)

![Logistic Regression Confusion Matrix](images/lr_cm_smote.png)

## 6. Run 2 – Decision Tree (with SMOTE)

In [8]:
with mlflow.start_run(run_name='DecisionTree_SMOTE') as run_dt:
    max_depth_val        = 12
    min_samples_split_val = 5
    criterion_val        = 'gini'

    dt = DecisionTreeClassifier(
        max_depth=max_depth_val,
        min_samples_split=min_samples_split_val,
        criterion=criterion_val,
        random_state=42
    )
    dt.fit(X_train_smote, y_train_smote)
    y_pred = dt.predict(X_test)
    y_proba = dt.predict_proba(X_test)[:, 1]

    acc   = accuracy_score(y_test, y_pred)
    f1    = f1_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, zero_division=0)
    rec   = recall_score(y_test, y_pred, zero_division=0)
    auc   = roc_auc_score(y_test, y_proba)

    mlflow.log_param('model_type',         'Decision Tree')
    mlflow.log_param('max_depth',          max_depth_val)
    mlflow.log_param('min_samples_split',  min_samples_split_val)
    mlflow.log_param('criterion',          criterion_val)
    mlflow.log_param('balancing',          'SMOTE')
    mlflow.log_param('train_size_before',  X_train.shape[0])
    mlflow.log_param('train_size_after',   X_train_smote.shape[0])

    mlflow.log_metric('accuracy',  acc)
    mlflow.log_metric('f1_score',  f1)
    mlflow.log_metric('precision', prec)
    mlflow.log_metric('recall',    rec)
    mlflow.log_metric('roc_auc',   auc)

    mlflow.set_tag('Algorithm',       'Decision Tree')
    mlflow.set_tag('Dataset_Version', DATASET_VERSION)
    mlflow.set_tag('Balancing',       'SMOTE')

    cm_path = plot_confusion_matrix(y_test, y_pred,
                                    'Confusion Matrix – Decision Tree (SMOTE)',
                                    'dt_cm_smote.png')
    mlflow.log_artifact(cm_path)

    sig = infer_signature(X_train_smote, dt.predict(X_train_smote))
    mlflow.sklearn.log_model(dt, 'decision_tree_model', signature=sig)

    print(f'DT (SMOTE)      |  acc={acc:.4f}  f1={f1:.4f}  recall={rec:.4f}  auc={auc:.4f}')

2026/04/19 07:34:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 07:34:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


DT (SMOTE)      |  acc=0.7284  f1=0.0688  recall=0.7583  auc=0.8128


### Confusion Matrix – Decision Tree (SMOTE)

![Decision Tree Confusion Matrix](images/dt_cm_smote.png)


In [9]:
with mlflow.start_run(run_name='OLS_SMOTE') as run_ols:
    X_train_sm = sm.add_constant(X_train_smote)
    X_test_sm  = sm.add_constant(X_test)

    ols = sm.OLS(y_train_smote, X_train_sm).fit()
    y_pred_cont = ols.predict(X_test_sm)
    y_pred = (y_pred_cont >= 0.5).astype(int)

    acc   = accuracy_score(y_test, y_pred)
    f1    = f1_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, zero_division=0)
    rec   = recall_score(y_test, y_pred, zero_division=0)
    r2    = ols.rsquared
    mse   = np.mean((y_test - y_pred_cont) ** 2)

    mlflow.log_param('model_type',     'OLS Regression')
    mlflow.log_param('threshold',      0.5)
    mlflow.log_param('n_features',     X_train_smote.shape[1])
    mlflow.log_param('balancing',      'SMOTE')
    mlflow.log_param('train_size_before', X_train.shape[0])
    mlflow.log_param('train_size_after',  X_train_smote.shape[0])

    mlflow.log_metric('accuracy',  acc)
    mlflow.log_metric('f1_score',  f1)
    mlflow.log_metric('precision', prec)
    mlflow.log_metric('recall',    rec)
    mlflow.log_metric('r_squared', r2)
    mlflow.log_metric('mse',       mse)

    mlflow.set_tag('Algorithm',       'OLS')
    mlflow.set_tag('Dataset_Version', DATASET_VERSION)
    mlflow.set_tag('Balancing',       'SMOTE')

    # BONUS: Confidence intervals
    alpha = 0.05
    conf_interval = ols.conf_int(alpha)
    conf_interval.columns = ['lower_95', 'upper_95']
    conf_interval.to_csv('conf_intervals_smote.csv')
    mlflow.log_artifact('conf_intervals_smote.csv')

    # Error distribution plot
    errors = y_test.values - y_pred_cont.values
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.histplot(errors, kde=True, ax=ax, color='steelblue')
    ax.set_title('Residual Distribution – OLS Regression (SMOTE)')
    ax.set_xlabel('Residual')
    plt.tight_layout()
    plt.savefig('ols_error_dist_smote.png', dpi=150)
    plt.close()
    mlflow.log_artifact('ols_error_dist_smote.png')

    cm_path = plot_confusion_matrix(y_test, y_pred,
                                    'Confusion Matrix – OLS Regression (SMOTE)',
                                    'ols_cm_smote.png')
    mlflow.log_artifact(cm_path)

    mlflow.statsmodels.log_model(ols, 'ols_regression_model')

    print(f'OLS (SMOTE)     |  acc={acc:.4f}  f1={f1:.4f}  recall={rec:.4f}  mse={mse:.4f}')

2026/04/19 07:34:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


OLS (SMOTE)     |  acc=0.5439  f1=0.0319  recall=0.5683  mse=0.2477


### Confusion Matrix – OLS Regression (SMOTE)

![OLS Confusion Matrix](images/ols_cm_smote.png)

### Residual Distribution – OLS Regression (SMOTE)

![OLS Residual Distribution](images/ols_error_dist_smote.png)


## 8. Model Comparison Summary

In [10]:
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=['metrics.f1_score DESC']
)

display_cols = [
    'tags.mlflow.runName', 'tags.Algorithm',
    'metrics.accuracy', 'metrics.f1_score',
    'metrics.precision', 'metrics.recall', 'metrics.roc_auc'
]
available = [c for c in display_cols if c in runs_df.columns]
print('\n=== Run Comparison (ordered by F1-Score) ===\n')
print(runs_df[available].to_string(index=False))


=== Run Comparison (ordered by F1-Score) ===

     tags.mlflow.runName      tags.Algorithm  metrics.accuracy  metrics.f1_score  metrics.precision  metrics.recall  metrics.roc_auc
      DecisionTree_SMOTE       Decision Tree          0.728399          0.068848           0.036061        0.758273         0.812753
LogisticRegression_SMOTE Logistic Regression          0.501010          0.040449           0.020753        0.794245         0.685656
               OLS_SMOTE                 OLS          0.543936          0.031950           0.016437        0.568345              NaN


### MLflow UI – Run Comparison

![MLflow Run Comparison](images/OLS_Smote_testrun_metrics.png)

In [11]:
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.f1_score DESC'],
    max_results=1
)
best_run = runs[0]
run_name = best_run.data.tags.get('mlflow.runName', '')
f1_best = best_run.data.metrics.get('f1_score', 'n/a')
print(f'\nBest run: {run_name}  (run_id={best_run.info.run_id})')
print(f'F1-score: {f1_best}')

artifact_map = {
    'LogisticRegression_SMOTE': 'logistic_regression_model',
    'DecisionTree_SMOTE':       'decision_tree_model',
    'OLS_SMOTE':                'ols_regression_model',
}
artifact_path = next(
    (v for k, v in artifact_map.items() if k in run_name), None
)

MODEL_NAME = 'PhishingDetectionBestModel'
if artifact_path:
    model_uri = f'runs:/{best_run.info.run_id}/{artifact_path}'
    print(f'Registering from: {model_uri}')
    reg = mlflow.register_model(model_uri, MODEL_NAME)
    print(f'✓ Registered as version {reg.version}')
else:
    print('ERROR: Could not determine artifact path')

Registered model 'PhishingDetectionBestModel' already exists. Creating a new version of this model...
2026/04/19 07:34:46 WARNING mlflow.tracking._model_registry.fluent: Run with id 8d5c41aee0fd42589c22cc1833ef047c has no artifacts at artifact path 'decision_tree_model', registering model based on models:/m-88c01c78456f4b35afca72bae6970c66 instead



Best run: DecisionTree_SMOTE  (run_id=8d5c41aee0fd42589c22cc1833ef047c)
F1-score: 0.06884838983604416
Registering from: runs:/8d5c41aee0fd42589c22cc1833ef047c/decision_tree_model
✓ Registered as version 2


Created version '2' of model 'PhishingDetectionBestModel'.


## 10. View Results in MLflow UI



```bash
python -m mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

Then open [http://127.0.0.1:5000](http://127.0.0.1:5000) in your browser.